In [1]:
# multi_api_query.py
import os
import requests
import cohere

# ---------------- API CONFIGS ----------------
HF_API_URL = "https://router.huggingface.co/v1/chat/completions"
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
HF_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OLLAMA_API_URL = "http://localhost:11434/api/chat"

MAX_LENGTH = 300

# ---------------- HELPER ----------------
def trim_response(text):
    if len(text) > MAX_LENGTH:
        return text[:MAX_LENGTH] + "... (too long)"
    return text

# ---------------- QUERY FUNCTIONS ----------------
def query_groq(prompt):
    try:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def query_huggingface(prompt):
    try:
        headers = {"Authorization": f"Bearer {HF_API_KEY}", "Content-Type": "application/json"}
        payload = {
            "model": "meta-llama/Llama-3.1-8B-Instruct:cerebras",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500,
            "temperature": 0.7
        }
        response = requests.post(HF_API_URL, headers=headers, json=payload, timeout=60)
        if response.status_code != 200:
            return f"Error {response.status_code}: {response.text}"
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Error: {str(e)}"

def query_cohere(prompt):
    try:
        client = cohere.ClientV2(COHERE_API_KEY)
        response = client.chat(
            model="command-r-08-2024",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        )
        return response.message.content[0].text
    except Exception as e:
        return f"Error: {str(e)}"

def query_ollama(prompt):
    try:
        payload = {
            "model": "llama3.2",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0.7, "num_predict": 500}
        }
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
        if response.status_code != 200:
            return f"Error {response.status_code}: {response.text}"
        return response.json()["message"]["content"]
    except requests.exceptions.ConnectionError:
        return "Error: Ollama is not running."
    except Exception as e:
        return f"Error: {str(e)}"

# ---------------- MAIN ----------------
print("=" * 50)
print("        Multi-API Query Program")
print("=" * 50)

user_prompt = input("Enter your prompt: ").strip()
print("\nQuerying all 4 providers simultaneously...\n")

providers = [
    ("Groq",        query_groq),
    ("HuggingFace", query_huggingface),
    ("Cohere",      query_cohere),
    ("Ollama",      query_ollama),
]

for name, func in providers:
    print(f"[ {name} ]")
    print("-" * 50)
    result = trim_response(func(user_prompt))
    print(result)
    print()

        Multi-API Query Program


Enter your prompt:  Explain artificial intelligence in simple words



Querying all 4 providers simultaneously...

[ Groq ]
--------------------------------------------------
Artificial intelligence (AI) is a way to make machines, like computers, think and learn like humans. It's like creating a robot that can do tasks on its own, without being told exactly what to do every time.

Imagine you have a toy box full of different colored balls. A regular computer would need t... (too long)

[ HuggingFace ]
--------------------------------------------------
Artificial Intelligence (AI) is like a super smart computer that can think and learn like a human being. It's a way to make machines do tasks that normally require intelligence, such as:

1. Recognizing objects (like pictures or voices)
2. Learning from data (like how to play a game or solve a probl... (too long)

[ Cohere ]
--------------------------------------------------
Artificial Intelligence, often shortened to AI, is like teaching a computer or a machine to think and make decisions like humans do, b